In [0]:
from pyspark.sql import functions as F

trips = spark.table("workspace.clean.trips_weather")

demand_by_hour = (trips
    .withColumn("day_num", F.dayofweek("pickup_hour"))
    .withColumn("day_name", F.date_format("pickup_hour", "E"))
    .withColumn("hour_of_day", F.hour("pickup_hour"))
    .groupBy("day_num", "day_name", "hour_of_day")
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
    .orderBy("day_num", "hour_of_day"))

demand_by_hour.write.mode("overwrite").saveAsTable("workspace.analytics.demand_by_hour")

print(spark.table("workspace.analytics.demand_by_hour").count())

168


In [0]:
from pyspark.sql import functions as F

def classify(df):
    return (df
        .withColumn("weather_condition",
            F.when(F.col("precipitation_mm") <= 0, "dry")
             .when(F.col("temperature_c") <= 0, "snow")
             .otherwise("rain"))
        .withColumn("temp_band",
            F.when(F.col("temperature_c") <= 0, "1_0_or_below")
             .when(F.col("temperature_c") < 5, "2_0_to_5")
             .when(F.col("temperature_c") < 10, "3_5_to_10")
             .otherwise("4_10_plus")))

trips = classify(spark.table("workspace.clean.trips_weather"))
hours = classify(spark.table("workspace.clean.weather_hourly"))
keys = ["weather_condition", "temp_band"]

trip_stats = trips.groupBy(keys).agg(
    F.count("*").alias("trips"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"))

hour_counts = hours.groupBy(keys).agg(F.count("*").alias("hours"))

demand_by_weather = (hour_counts.join(trip_stats, keys, "left")
    .fillna(0, subset=["trips"])
    .withColumn("trips_per_hour", F.round(F.col("trips") / F.col("hours"), 2))
    .orderBy(keys))

demand_by_weather.write.mode("overwrite").saveAsTable("workspace.analytics.demand_by_weather")

assert demand_by_weather.agg(F.sum("trips")).first()[0] == trips.count()
display(spark.table("workspace.analytics.demand_by_weather"))

weather_condition,temp_band,hours,trips,avg_fare,avg_distance_miles,trips_per_hour
dry,1_0_or_below,657,9353,12.23,2.86,14.24
dry,2_0_to_5,390,6224,12.46,2.89,15.96
dry,3_5_to_10,155,2722,12.23,2.83,17.56
dry,4_10_plus,51,859,12.55,2.91,16.84
rain,2_0_to_5,66,993,12.61,2.99,15.05
rain,3_5_to_10,19,287,13.37,3.12,15.11
rain,4_10_plus,43,709,12.54,2.82,16.49
snow,1_0_or_below,59,700,11.0,2.5,11.86
